# Notebook 3: NER with Transformers (BERT)

Transformer models like **BERT** achieve state-of-the-art NER performance by leveraging deep contextual embeddings.

## What you'll learn:
1. How transformers approach NER (token classification)
2. Using Hugging Face's NER pipeline (zero-code inference)
3. Understanding tokenization and label alignment
4. Fine-tuning BERT on the CoNLL-2003 NER dataset
5. Evaluating with seqeval metrics

## 1. How Transformers Do NER

NER with transformers is framed as **token classification**:

```
Input tokens:  [CLS] Steve Jobs founded Apple [SEP]
Labels:        -100  B-PER I-PER O       B-ORG -100
```

Key differences from spaCy:
- **Subword tokenization**: "Jobs" might become ["Job", "##s"] — only the first subword gets a label
- **Special tokens**: [CLS] and [SEP] get label `-100` (ignored in loss)
- **Contextual embeddings**: Each token's representation considers the entire sentence

## 2. Setup

In [ ]:
# !pip install transformers datasets torch seqeval

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    pipeline,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from datasets import load_dataset
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 3. Zero-Code NER with Hugging Face Pipeline

The fastest way to get started — use a pre-trained NER model directly.

In [ ]:
# Load a pre-trained NER pipeline
ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"  # Groups B-/I- tokens into full entities
)

text = "Elon Musk founded SpaceX in Hawthorne, California and serves as its CEO."
results = ner_pipeline(text)

print(f"Text: {text}\n")
print(f"{'Entity':25s} {'Label':10s} {'Score':8s}")
print("-" * 45)
for entity in results:
    print(f"{entity['word']:25s} {entity['entity_group']:10s} {entity['score']:.4f}")

In [ ]:
# Try multiple sentences
texts = [
    "Barack Obama was the 44th president of the United States.",
    "Google acquired DeepMind for $500 million in January 2014.",
    "The Eiffel Tower in Paris attracts 7 million visitors annually.",
]

for text in texts:
    results = ner_pipeline(text)
    print(f"\n> {text}")
    for entity in results:
        print(f"  [{entity['entity_group']:6s}] {entity['word']:20s} (score: {entity['score']:.3f})")

## 4. Understanding Subword Tokenization

BERT uses **WordPiece** tokenization which splits unknown words into subwords.
This is crucial for NER because we need to align labels with subword tokens.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# See how BERT tokenizes text
text = "Steve Jobs co-founded Apple in Cupertino"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print(f"Original text: {text}")
print(f"\nTokens: {tokens}")
print(f"Token IDs: {token_ids}")
print(f"\nNotice how 'co-founded' becomes multiple subword tokens!")
print(f"The '##' prefix means 'continuation of previous word'.")

# Word-to-token alignment
encoding = tokenizer(text.split(), is_split_into_words=True, return_offsets_mapping=False)
word_ids = encoding.word_ids()
print(f"\nWord IDs (maps each token to its original word index):")
print(f"Tokens:   {tokenizer.convert_ids_to_tokens(encoding['input_ids'])}")
print(f"Word IDs: {word_ids}")

## 5. Loading the CoNLL-2003 Dataset

**CoNLL-2003** is the standard benchmark for NER. It contains:
- 4 entity types: PER, ORG, LOC, MISC
- ~14K training sentences from Reuters news
- BIO tagging scheme

In [ ]:
# Load CoNLL-2003
dataset = load_dataset("conll2003", trust_remote_code=True)

print("Dataset splits:")
for split, data in dataset.items():
    print(f"  {split}: {len(data)} examples")

# Look at label names
label_names = dataset["train"].features["ner_tags"].feature.names
print(f"\nNER labels: {label_names}")

# Show a sample
sample = dataset["train"][0]
print(f"\nSample sentence:")
print(f"  Tokens: {sample['tokens']}")
print(f"  NER tags: {sample['ner_tags']}")
print(f"  Labels: {[label_names[t] for t in sample['ner_tags']]}")

## 6. Tokenize and Align Labels

The key challenge: BERT's subword tokens don't match the original word-level tokens.
We need to **align** labels so each subword gets the correct label.

Strategy:
- First subword of a word -> gets the word's label
- Subsequent subwords -> get `-100` (ignored) OR the `I-` version of the label
- Special tokens ([CLS], [SEP]) -> get `-100`

In [ ]:
model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    """Tokenize inputs and align NER labels with subword tokens."""
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
    )
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens get -100
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First subword of a new word -> use the word's label
                label_ids.append(label[word_idx])
            else:
                # Subsequent subwords -> ignore
                label_ids.append(-100)
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply to the entire dataset
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

print("Tokenization complete!")
print(f"Tokenized train size: {len(tokenized_datasets['train'])}")

## 7. Fine-Tune BERT for NER

Now we fine-tune `bert-base-cased` on CoNLL-2003.

**Note**: This will take a few minutes on CPU, or ~2 min on GPU. We use a small subset for demonstration.

In [ ]:
from seqeval.metrics import classification_report, f1_score

# Define label mappings
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

# Load BERT with a token classification head
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

print(f"Model loaded: {model_checkpoint}")
print(f"Number of labels: {len(label_names)}")
print(f"Labels: {label_names}")
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# Metric function for evaluation during training
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    
    # Remove padding (-100) and convert IDs to label names
    true_labels = []
    true_predictions = []
    
    for prediction, label in zip(predictions, labels):
        true_label = []
        true_pred = []
        for p, l in zip(prediction, label):
            if l != -100:
                true_label.append(id2label[l])
                true_pred.append(id2label[p])
        true_labels.append(true_label)
        true_predictions.append(true_pred)
    
    f1 = f1_score(true_labels, true_predictions)
    return {"f1": f1}

print("Metric function defined.")

In [ ]:
# Use a small subset for quick demonstration (remove .select() for full training)
small_train = tokenized_datasets["train"].select(range(1000))
small_eval = tokenized_datasets["validation"].select(range(500))

# Training arguments
training_args = TrainingArguments(
    output_dir="../output/bert-ner-conll2003",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer configured. Starting training...")
print(f"Training samples: {len(small_train)}, Eval samples: {len(small_eval)}")

In [ ]:
# Train!
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Training loss: {train_result.training_loss:.4f}")

## 8. Detailed Evaluation

In [ ]:
# Get predictions on the eval set
predictions, labels, _ = trainer.predict(small_eval)
predictions = np.argmax(predictions, axis=-1)

# Convert to label names (removing -100 padding)
true_labels = []
true_predictions = []

for prediction, label in zip(predictions, labels):
    true_label = []
    true_pred = []
    for p, l in zip(prediction, label):
        if l != -100:
            true_label.append(id2label[l])
            true_pred.append(id2label[p])
    true_labels.append(true_label)
    true_predictions.append(true_pred)

# Detailed classification report
print(classification_report(true_labels, true_predictions))

## 9. Inference with Fine-Tuned Model

In [ ]:
# Use the fine-tuned model for inference
fine_tuned_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1,
)

test_texts = [
    "Tim Cook announced Apple's new headquarters in Austin, Texas.",
    "The United Nations held a summit in Geneva, Switzerland last Monday.",
    "Satya Nadella led Microsoft's acquisition of Activision Blizzard for $69 billion.",
]

for text in test_texts:
    results = fine_tuned_pipeline(text)
    print(f"\n> {text}")
    for entity in results:
        print(f"  [{entity['entity_group']:6s}] {entity['word']:25s} (score: {entity['score']:.3f})")

## 10. Comparison: spaCy vs BERT for NER

| Aspect | spaCy | BERT (Transformers) |
|--------|-------|--------------------|
| **Speed** | Fast (CPU-friendly) | Slower (benefits from GPU) |
| **Accuracy** | Good | State-of-the-art |
| **Training data needed** | Less (~200 examples) | More (~1000+ examples) |
| **Model size** | Small (~15 MB) | Large (~400 MB) |
| **Custom entities** | Easy to add | Requires fine-tuning |
| **Best for** | Production, real-time | Research, high accuracy |
| **Tokenization** | Word-level | Subword (WordPiece) |

## 11. Summary and Next Steps

### What we covered:
1. Transformer-based NER as token classification
2. Using pre-trained NER models via Hugging Face pipelines
3. Subword tokenization and label alignment
4. Fine-tuning BERT on CoNLL-2003
5. Evaluation with seqeval

### To go further:
- **Full training**: Remove the `.select()` subset to train on full CoNLL-2003
- **Larger models**: Try `bert-large-cased` or `roberta-large`
- **Custom data**: Fine-tune on your own domain-specific NER data
- **Few-shot NER**: Explore models like `SpanBERT` or `GLiNER` for low-resource settings
- **Production**: Use ONNX or TorchScript to optimize for deployment